# Monthly Sales Revenue and Volume

## Purpose and approach
This notebook summarizes sales in **`SH.SALES` by calendar month**, with implementations in Python, R, and Oracle SQL. Aggregation takes place inside Oracle Database; only bounded monthly results are returned for display, rather than individual sales records.

| Measure | Definition | Interpretation |
|---|---|---|
| Sales revenue | `SUM(AMOUNT_SOLD)` | Monthly revenue; currency is unspecified |
| Sales-row volume | `COUNT(*)` | Number of sales records, used as an order-volume proxy |
| Units sold | `SUM(QUANTITY_SOLD)` | Monthly product-unit volume |

**Sales-row volume is not a distinct order count:** the analysis identifies no order identifier in `SH.SALES`.

The source groups dates using `TRUNC(TIME_ID, 'MM')` and presents months chronologically. Only months with recorded sales are included; missing months are not filled with zeros. Display is bounded to the latest **1,200 recorded months**.

## Recorded results
The saved SQL output reports **48 recorded months**. Examples visible in both the R and SQL outputs include:

- **January 1998:** revenue of **2,277,420.49**, with **15,132 sales rows** and **15,132 units sold**.
- **February 1998:** revenue of **2,372,690.87**, with **14,307 sales rows** and **14,307 units sold**.

These examples show that higher revenue need not coincide with higher sales-row volume. They do not establish a longer-term trend or its causes.

## Evidence and limitations
All three analysis cells have persisted outputs, with no errors shown in the supplied result evidence. However, portions of the source and result tables are omitted, limiting a complete trend summary. The Python text evidence contains a formatted-table object reference rather than its numeric contents; the R and SQL excerpts provide the figures above.

Saved outputs document earlier displayed results, not the current database contents or in-memory kernel state.

In [1]:
import oml
from IPython.display import display

if not oml.isconnected():
    raise RuntimeError("An authenticated OML database session is required.")

# Oracle performs all aggregation; only up to 1,200 monthly results are downloaded.
monthly_sales_sql = """
SELECT SALES_MONTH, SALES_REVENUE, SALES_ROW_VOLUME, UNITS_SOLD, TOTAL_MONTHS
FROM (
    SELECT TO_CHAR(TRUNC(TIME_ID, 'MM'), 'YYYY-MM') AS SALES_MONTH,
           SUM(AMOUNT_SOLD) AS SALES_REVENUE,
           COUNT(*) AS SALES_ROW_VOLUME,
           SUM(QUANTITY_SOLD) AS UNITS_SOLD,
           COUNT(*) OVER () AS TOTAL_MONTHS
    FROM SH.SALES
    GROUP BY TRUNC(TIME_ID, 'MM')
    ORDER BY TRUNC(TIME_ID, 'MM') DESC
    FETCH FIRST 1200 ROWS ONLY
)
ORDER BY SALES_MONTH
"""

monthly_sales = oml.sync(query=monthly_sales_sql).pull()

print("Monthly sales revenue and volume — SH.SALES")
print(
    "SH.SALES has no order identifier. SALES_ROW_VOLUME counts sales rows "
    "as a volume proxy, not distinct orders; UNITS_SOLD counts product units."
)
print("Revenue is SUM(AMOUNT_SOLD); currency is not specified in the source metadata.")
print("Only months with recorded sales are included.")

if monthly_sales.empty:
    print("No sales data was returned.")
else:
    total_months = int(monthly_sales["TOTAL_MONTHS"].iloc[0])
    if total_months > len(monthly_sales):
        print(
            f"Showing the latest {len(monthly_sales):,} of "
            f"{total_months:,} recorded months, in chronological order."
        )
    # Local formatting is limited to the bounded, database-aggregated results.
    display(
        monthly_sales.drop(columns=["TOTAL_MONTHS"]).style.format(
            {
                "SALES_REVENUE": "{:,.2f}",
                "SALES_ROW_VOLUME": "{:,.0f}",
                "UNITS_SOLD": "{:,.0f}",
            },
            na_rep="NULL",
        )
    )

Monthly sales revenue and volume — SH.SALES
SH.SALES has no order identifier. SALES_ROW_VOLUME counts sales rows as a volume proxy, not distinct orders; UNITS_SOLD counts product units.
Revenue is SUM(AMOUNT_SOLD); currency is not specified in the source metadata.
Only months with recorded sales are included.


,SALES_MONTH,SALES_REVENUE,SALES_ROW_VOLUME,UNITS_SOLD
0,1998-01,"2,277,420.49","15,132","15,132"
1,1998-02,"2,372,690.87","14,307","14,307"
2,1998-03,"1,830,572.64","14,248","14,248"
3,1998-04,"1,975,978.30","11,818","11,818"
4,1998-05,"1,748,287.23","12,309","12,309"
5,1998-06,"1,869,728.61","11,631","11,631"
6,1998-07,"1,932,282.28","16,257","16,257"
7,1998-08,"1,972,532.63","17,199","17,199"
8,1998-09,"2,167,008.19","17,059","17,059"
9,1998-10,"2,236,464.53","19,878","19,878"


In [3]:
local({
    if (!requireNamespace("ORE", quietly = TRUE)) {
        stop("OML4R (ORE) must be available in the active R kernel.")
    }
    if (!requireNamespace("DBI", quietly = TRUE)) {
        stop("DBI must be available to query the existing Oracle session.")
    }
    # ORE is an umbrella package; connection functions belong to OREbase.
    if (!requireNamespace("OREbase", quietly = TRUE)) {
        stop("OML4R's OREbase package must be available.")
    }
    if (!OREbase::ore.is.connected()) {
        stop("An authenticated OML4R database session is required.")
    }
    if (!requireNamespace("ROracle", quietly = TRUE)) {
        stop("ROracle must be available to reuse the existing Oracle session.")
    }

    # No documented OML4R connection accessor was verified. Use the public
    # DBI/ROracle interface to enumerate existing connections, without opening
    # or closing any connection. Refuse ambiguous session selection.
    connections <- DBI::dbListConnections(ROracle::Oracle())
    if (length(connections) != 1L) {
        stop(sprintf(
            paste0(
                "Expected exactly one existing ROracle connection; found %d. ",
                "Cannot safely select the authenticated session."
            ),
            length(connections)
        ))
    }
    connection <- connections[[1L]]

    # Oracle executes the read-only aggregation; only bounded monthly
    # aggregates, never transaction rows, enter R memory.

    sql <- "
        SELECT SALES_MONTH, SALES_REVENUE, SALES_ROW_VOLUME,
               UNITS_SOLD, TOTAL_MONTHS
        FROM (
            SELECT TO_CHAR(TRUNC(TIME_ID, 'MM'), 'YYYY-MM') AS SALES_MONTH,
                   SUM(AMOUNT_SOLD) AS SALES_REVENUE,
                   COUNT(*) AS SALES_ROW_VOLUME,
                   SUM(QUANTITY_SOLD) AS UNITS_SOLD,
                   COUNT(*) OVER () AS TOTAL_MONTHS
            FROM SH.SALES
            GROUP BY TRUNC(TIME_ID, 'MM')
            ORDER BY TRUNC(TIME_ID, 'MM') DESC
            FETCH FIRST 1200 ROWS ONLY
        )
        ORDER BY SALES_MONTH
    "

    monthly_sales <- DBI::dbGetQuery(connection, sql)

    cat("Monthly sales revenue and volume — SH.SALES
")
    cat(
        "SH.SALES has no order identifier. SALES_ROW_VOLUME counts sales rows
",
        "as an order-volume proxy, not distinct orders. UNITS_SOLD counts units.
",
        sep = ""
    )
    cat("Revenue is SUM(AMOUNT_SOLD); currency is unspecified.
")
    cat("Only months with recorded sales are included.

")

    if (nrow(monthly_sales) == 0L) {
        cat("No sales data was returned.
")
    } else {
        total_months <- as.numeric(monthly_sales$TOTAL_MONTHS[1L])
        if (total_months > nrow(monthly_sales)) {
            cat(sprintf(
                "Showing the latest %d of %.0f recorded months, chronologically.
",
                nrow(monthly_sales), total_months
            ))
        }

        # Formatting applies only to the bounded, already aggregated results.
        display_table <- monthly_sales[
            , c("SALES_MONTH", "SALES_REVENUE",
                "SALES_ROW_VOLUME", "UNITS_SOLD"),
            drop = FALSE
        ]
        display_table$SALES_REVENUE <- formatC(
            display_table$SALES_REVENUE,
            format = "f", digits = 2, big.mark = ","
        )
        for (column in c("SALES_ROW_VOLUME", "UNITS_SOLD")) {
            display_table[[column]] <- format(
                display_table[[column]],
                scientific = FALSE, trim = TRUE, big.mark = ","
            )
        }
        print(display_table, row.names = FALSE, max = 10000L)
    }
})

Monthly sales revenue and volume — SH.SALES
SH.SALES has no order identifier. SALES_ROW_VOLUME counts sales rows
as an order-volume proxy, not distinct orders. UNITS_SOLD counts units.
Revenue is SUM(AMOUNT_SOLD); currency is unspecified.
Only months with recorded sales are included.

 SALES_MONTH SALES_REVENUE SALES_ROW_VOLUME UNITS_SOLD
     1998-01  2,277,420.49           15,132     15,132
     1998-02  2,372,690.87           14,307     14,307
     1998-03  1,830,572.64           14,248     14,248
     1998-04  1,975,978.30           11,818     11,818
     1998-05  1,748,287.23           12,309     12,309
     1998-06  1,869,728.61           11,631     11,631
     1998-07  1,932,282.28           16,257     16,257
     1998-08  1,972,532.63           17,199     17,199
     1998-09  2,167,008.19           17,059     17,059
     1998-10  2,236,464.53           19,878     19,878
     1998-11  1,959,664.22           14,612     14,612
     1998-12  1,741,284.96           14,384     14,384

In [4]:
-- Read-only, in-database aggregation; no database objects are created.
SELECT
    'SH.SALES has no order identifier. SALES_ROW_VOLUME counts sales rows, '
    || 'not distinct orders; UNITS_SOLD counts product units. '
    || 'Revenue currency is unspecified. Only recorded months are included. '
    || 'At most the latest 1,200 months are shown.' AS METRIC_NOTES
FROM DUAL;

-- Numeric measures remain chart-ready. This kernel renders tables, not charts.
WITH monthly_sales AS (
    SELECT
        TRUNC(TIME_ID, 'MM') AS MONTH_START,
        SUM(AMOUNT_SOLD) AS SALES_REVENUE,
        COUNT(*) AS SALES_ROW_VOLUME,
        SUM(QUANTITY_SOLD) AS UNITS_SOLD
    FROM SH.SALES
    GROUP BY TRUNC(TIME_ID, 'MM')
),
bounded_months AS (
    SELECT
        MONTH_START,
        SALES_REVENUE,
        SALES_ROW_VOLUME,
        UNITS_SOLD,
        COUNT(*) OVER () AS TOTAL_RECORDED_MONTHS
    FROM monthly_sales
    ORDER BY MONTH_START DESC NULLS LAST
    FETCH FIRST 1200 ROWS ONLY
)
SELECT
    TO_CHAR(MONTH_START, 'YYYY-MM') AS SALES_MONTH,
    SALES_REVENUE,
    SALES_ROW_VOLUME,
    UNITS_SOLD,
    TOTAL_RECORDED_MONTHS
FROM bounded_months
ORDER BY MONTH_START NULLS LAST;

METRIC_NOTES
"SH.SALES has no order identifier. SALES_ROW_VOLUME counts sales rows, not distinct orders; UNITS_SOLD counts product units. Revenue currency is unspecified. Only recorded months are included. At most the latest 1,200 months are shown."


SALES_MONTH,SALES_REVENUE,SALES_ROW_VOLUME,UNITS_SOLD,TOTAL_RECORDED_MONTHS
1998-01,2277420.49,15132,15132,48
1998-02,2372690.87,14307,14307,48
1998-03,1830572.64,14248,14248,48
1998-04,1975978.3,11818,11818,48
1998-05,1748287.23,12309,12309,48
1998-06,1869728.61,11631,11631,48
1998-07,1932282.28,16257,16257,48
1998-08,1972532.63,17199,17199,48
1998-09,2167008.19,17059,17059,48
1998-10,2236464.53,19878,19878,48
